In [1]:
import numpy as np
import pandas as pd
from typing import Tuple

print("✅ Imports loaded")

✅ Imports loaded


## 1. Metric Functions (from main_experiment_grid_search_1229.ipynb)

In [2]:
def rmse_mad_on_test(pred: pd.DataFrame, test: pd.DataFrame) -> Tuple[float, float]:
    """Calculate RMSE and MAD on test set."""
    P = pred.values
    T = test.values
    mask = ~np.isnan(T)
    if not mask.any():
        return np.nan, np.nan
    diff = P[mask] - T[mask]
    rmse = float(np.sqrt(np.mean(diff**2)))
    mad = float(np.mean(np.abs(diff)))
    return rmse, mad


def precision_recall_at_n(
    pred: pd.DataFrame, train: pd.DataFrame, test: pd.DataFrame,
    N: int, relevance_threshold: float = 4.0
) -> Tuple[float, float]:
    """
    Calculate Precision@N and Recall@N.
    Candidate items: not in train & observed in test
    Relevant items: test rating >= relevance_threshold
    """
    precisions, recalls = [], []
    
    for u_idx in range(pred.shape[1]):
        train_col = train.iloc[:, u_idx]
        test_col = test.iloc[:, u_idx]
        pred_col = pred.iloc[:, u_idx]
        
        cand_mask = train_col.isna() & ~test_col.isna()
        if not cand_mask.any():
            continue
        
        relevant_mask = (test_col >= relevance_threshold) & cand_mask
        n_rel = int(relevant_mask.sum())
        
        scores = pred_col[cand_mask]
        topN_items = scores.sort_values(ascending=False).head(N).index
        
        n_hit = int(relevant_mask.loc[topN_items].sum()) if len(topN_items) else 0
        
        prec_u = n_hit / min(N, int(cand_mask.sum())) if int(cand_mask.sum()) > 0 else np.nan
        rec_u = n_hit / n_rel if n_rel > 0 else np.nan
        
        if not np.isnan(prec_u): 
            precisions.append(prec_u)
        if not np.isnan(rec_u):  
            recalls.append(rec_u)
    
    precision = float(np.mean(precisions)) if precisions else np.nan
    recall = float(np.mean(recalls)) if recalls else np.nan
    return precision, recall

print("✅ Metric functions defined")

✅ Metric functions defined


## 2. Test RMSE and MAD

**Scenario:**
- Predictions: [3.0, 4.0, 5.0]
- Ground truth: [3.5, 4.5, 4.0]

**Expected:**
- Errors: [-0.5, -0.5, 1.0]
- RMSE = sqrt(mean([0.25, 0.25, 1.0])) = sqrt(0.5) ≈ 0.707
- MAD = mean([0.5, 0.5, 1.0]) = 0.667

In [3]:
# Create toy example for RMSE/MAD
pred_rmse = pd.DataFrame(
    [[3.0, np.nan],
     [4.0, np.nan],
     [5.0, np.nan]],
    index=['item1', 'item2', 'item3'],
    columns=['user1', 'user2']
)

test_rmse = pd.DataFrame(
    [[3.5, np.nan],
     [4.5, np.nan],
     [4.0, np.nan]],
    index=['item1', 'item2', 'item3'],
    columns=['user1', 'user2']
)

rmse, mad = rmse_mad_on_test(pred_rmse, test_rmse)

print("\n" + "="*60)
print("RMSE/MAD Test")
print("="*60)
print(f"Predictions:    [3.0, 4.0, 5.0]")
print(f"Ground Truth:   [3.5, 4.5, 4.0]")
print(f"Errors:         [-0.5, -0.5, 1.0]")
print(f"\nExpected RMSE:  0.707")
print(f"Computed RMSE:  {rmse:.3f}")
print(f"✓ Match: {abs(rmse - 0.707) < 0.01}")
print(f"\nExpected MAD:   0.667")
print(f"Computed MAD:   {mad:.3f}")
print(f"✓ Match: {abs(mad - 0.667) < 0.01}")

# Verify manually
errors = np.array([-0.5, -0.5, 1.0])
expected_rmse = np.sqrt(np.mean(errors**2))
expected_mad = np.mean(np.abs(errors))
print(f"\nManual calculation:")
print(f"  RMSE = sqrt(mean([0.25, 0.25, 1.0])) = {expected_rmse:.3f}")
print(f"  MAD  = mean([0.5, 0.5, 1.0]) = {expected_mad:.3f}")
print(f"\n✅ RMSE and MAD are correctly implemented!")


RMSE/MAD Test
Predictions:    [3.0, 4.0, 5.0]
Ground Truth:   [3.5, 4.5, 4.0]
Errors:         [-0.5, -0.5, 1.0]

Expected RMSE:  0.707
Computed RMSE:  0.707
✓ Match: True

Expected MAD:   0.667
Computed MAD:   0.667
✓ Match: True

Manual calculation:
  RMSE = sqrt(mean([0.25, 0.25, 1.0])) = 0.707
  MAD  = mean([0.5, 0.5, 1.0]) = 0.667

✅ RMSE and MAD are correctly implemented!


## 3. Test Precision@N and Recall@N

**Scenario for User 1:**
- Train: [5.0, NaN, NaN, NaN, NaN] (only item1 rated)
- Test:  [NaN, 5.0, 4.0, 3.0, 2.0] (items 2-5)
- Pred:  [5.0, 4.8, 4.5, 3.8, 3.2] (sorted descending in candidates)
- Threshold: 4.0

**Analysis:**
- Candidates: items 2-5 (not in train & in test)
- Relevant items (≥4.0): item2(5.0), item3(4.0) → 2 items
- Top-3 predictions: item2(4.8), item3(4.5), item4(3.8)
- Hits in Top-3: item2 ✓, item3 ✓ → 2 hits

**Expected @N=3:**
- Precision@3 = hits / N = 2/3 = 0.667
- Recall@3 = hits / total_relevant = 2/2 = 1.0

In [4]:
# Create toy example for Precision/Recall
train_pr = pd.DataFrame(
    [[5.0, np.nan],   # item1: in train for user1
     [np.nan, np.nan], # item2: candidate
     [np.nan, np.nan], # item3: candidate
     [np.nan, np.nan], # item4: candidate
     [np.nan, np.nan]], # item5: candidate
    index=['item1', 'item2', 'item3', 'item4', 'item5'],
    columns=['user1', 'user2']
)

test_pr = pd.DataFrame(
    [[np.nan, np.nan],
     [5.0, np.nan],    # item2: relevant (≥4.0)
     [4.0, np.nan],    # item3: relevant (≥4.0)
     [3.0, np.nan],    # item4: not relevant
     [2.0, np.nan]],   # item5: not relevant
    index=['item1', 'item2', 'item3', 'item4', 'item5'],
    columns=['user1', 'user2']
)

pred_pr = pd.DataFrame(
    [[5.0, np.nan],
     [4.8, np.nan],    # item2: highest predicted (relevant)
     [4.5, np.nan],    # item3: 2nd highest (relevant)
     [3.8, np.nan],    # item4: 3rd highest (not relevant)
     [3.2, np.nan]],   # item5: 4th highest (not relevant)
    index=['item1', 'item2', 'item3', 'item4', 'item5'],
    columns=['user1', 'user2']
)

N = 3
precision, recall = precision_recall_at_n(pred_pr, train_pr, test_pr, N=N, relevance_threshold=4.0)

print("\n" + "="*60)
print(f"Precision@{N} and Recall@{N} Test")
print("="*60)
print("\nUser 1 Analysis:")
print("  Train items:      [item1]")
print("  Test items:       [item2(5.0), item3(4.0), item4(3.0), item5(2.0)]")
print("  Candidate items:  [item2, item3, item4, item5] (not in train)")
print("  Relevant items:   [item2, item3] (rating ≥ 4.0) → 2 items")
print(f"\n  Predicted scores (candidates only):")
print(f"    item2: 4.8  ← rank 1 (relevant ✓)")
print(f"    item3: 4.5  ← rank 2 (relevant ✓)")
print(f"    item4: 3.8  ← rank 3 (not relevant ✗)")
print(f"    item5: 3.2  ← rank 4 (not relevant ✗)")
print(f"\n  Top-{N} recommendations: [item2, item3, item4]")
print(f"  Hits in Top-{N}:         [item2 ✓, item3 ✓] → 2 hits")

expected_precision = 2 / 3  # 2 hits / 3 recommendations
expected_recall = 2 / 2     # 2 hits / 2 relevant items

print(f"\nExpected Precision@{N}: {expected_precision:.3f} (2 hits / {N} recommended)")
print(f"Computed Precision@{N}: {precision:.3f}")
print(f"✓ Match: {abs(precision - expected_precision) < 0.01}")

print(f"\nExpected Recall@{N}:    {expected_recall:.3f} (2 hits / 2 relevant)")
print(f"Computed Recall@{N}:    {recall:.3f}")
print(f"✓ Match: {abs(recall - expected_recall) < 0.01}")

print(f"\n✅ Precision@{N} and Recall@{N} are correctly implemented!")


Precision@3 and Recall@3 Test

User 1 Analysis:
  Train items:      [item1]
  Test items:       [item2(5.0), item3(4.0), item4(3.0), item5(2.0)]
  Candidate items:  [item2, item3, item4, item5] (not in train)
  Relevant items:   [item2, item3] (rating ≥ 4.0) → 2 items

  Predicted scores (candidates only):
    item2: 4.8  ← rank 1 (relevant ✓)
    item3: 4.5  ← rank 2 (relevant ✓)
    item4: 3.8  ← rank 3 (not relevant ✗)
    item5: 3.2  ← rank 4 (not relevant ✗)

  Top-3 recommendations: [item2, item3, item4]
  Hits in Top-3:         [item2 ✓, item3 ✓] → 2 hits

Expected Precision@3: 0.667 (2 hits / 3 recommended)
Computed Precision@3: 0.667
✓ Match: True

Expected Recall@3:    1.000 (2 hits / 2 relevant)
Computed Recall@3:    1.000
✓ Match: True

✅ Precision@3 and Recall@3 are correctly implemented!


## 4. Edge Case: Precision when N > candidates

In [5]:
# Test when N=10 but only 4 candidates exist
N_large = 10
precision_large, recall_large = precision_recall_at_n(pred_pr, train_pr, test_pr, N=N_large, relevance_threshold=4.0)

print("\n" + "="*60)
print(f"Edge Case: N={N_large} > candidates(4)")
print("="*60)
print(f"\nCandidates: 4 items (item2, item3, item4, item5)")
print(f"Relevant:   2 items (item2, item3)")
print(f"Top-{N_large}: All 4 candidates (since N > candidates)")
print(f"Hits:      2 (item2, item3)")

# Precision should be hits / min(N, candidates) = 2/4 = 0.5
expected_precision_large = 2 / 4
expected_recall_large = 2 / 2  # Still 100% recall

print(f"\nExpected Precision@{N_large}: {expected_precision_large:.3f} (2 hits / 4 candidates)")
print(f"Computed Precision@{N_large}: {precision_large:.3f}")
print(f"✓ Match: {abs(precision_large - expected_precision_large) < 0.01}")

print(f"\nExpected Recall@{N_large}:    {expected_recall_large:.3f}")
print(f"Computed Recall@{N_large}:    {recall_large:.3f}")
print(f"✓ Match: {abs(recall_large - expected_recall_large) < 0.01}")

print(f"\n✅ Edge case handled correctly!")


Edge Case: N=10 > candidates(4)

Candidates: 4 items (item2, item3, item4, item5)
Relevant:   2 items (item2, item3)
Top-10: All 4 candidates (since N > candidates)
Hits:      2 (item2, item3)

Expected Precision@10: 0.500 (2 hits / 4 candidates)
Computed Precision@10: 0.500
✓ Match: True

Expected Recall@10:    1.000
Computed Recall@10:    1.000
✓ Match: True

✅ Edge case handled correctly!


## 5. Multi-User Test

In [6]:
# Create example with 2 users
train_multi = pd.DataFrame(
    [[5.0, 4.0],      # item1: both users rated in train
     [np.nan, np.nan], # item2: candidate for both
     [np.nan, np.nan], # item3: candidate for both
     [np.nan, np.nan]], # item4: candidate for both
    index=['item1', 'item2', 'item3', 'item4'],
    columns=['user1', 'user2']
)

test_multi = pd.DataFrame(
    [[np.nan, np.nan],
     [5.0, 3.0],       # item2: relevant for user1, not for user2
     [4.0, 5.0],       # item3: relevant for both
     [2.0, 4.0]],      # item4: not relevant for user1, relevant for user2
    index=['item1', 'item2', 'item3', 'item4'],
    columns=['user1', 'user2']
)

pred_multi = pd.DataFrame(
    [[5.0, 4.0],
     [4.9, 4.2],       # item2: rank1 for user1, rank2 for user2
     [4.5, 4.8],       # item3: rank2 for user1, rank1 for user2
     [3.0, 4.0]],      # item4: rank3 for user1, rank3 for user2
    index=['item1', 'item2', 'item3', 'item4'],
    columns=['user1', 'user2']
)

N_multi = 2
precision_multi, recall_multi = precision_recall_at_n(pred_multi, train_multi, test_multi, N=N_multi, relevance_threshold=4.0)

print("\n" + "="*60)
print(f"Multi-User Test: Precision@{N_multi} and Recall@{N_multi}")
print("="*60)

print("\nUser 1:")
print("  Relevant items:  [item2(5.0), item3(4.0)] → 2 items")
print("  Top-2 predicted: [item2(4.9), item3(4.5)]")
print("  Hits:           2")
print("  Precision@2:    2/2 = 1.0")
print("  Recall@2:       2/2 = 1.0")

print("\nUser 2:")
print("  Relevant items:  [item3(5.0), item4(4.0)] → 2 items")
print("  Top-2 predicted: [item3(4.8), item2(4.2)]")
print("  Hits:           1 (only item3)")
print("  Precision@2:    1/2 = 0.5")
print("  Recall@2:       1/2 = 0.5")

# Average across users
expected_precision_multi = (1.0 + 0.5) / 2  # 0.75
expected_recall_multi = (1.0 + 0.5) / 2     # 0.75

print(f"\nExpected Avg Precision@{N_multi}: {expected_precision_multi:.3f}")
print(f"Computed Avg Precision@{N_multi}: {precision_multi:.3f}")
print(f"✓ Match: {abs(precision_multi - expected_precision_multi) < 0.01}")

print(f"\nExpected Avg Recall@{N_multi}:    {expected_recall_multi:.3f}")
print(f"Computed Avg Recall@{N_multi}:    {recall_multi:.3f}")
print(f"✓ Match: {abs(recall_multi - expected_recall_multi) < 0.01}")

print(f"\n✅ Multi-user averaging works correctly!")


Multi-User Test: Precision@2 and Recall@2

User 1:
  Relevant items:  [item2(5.0), item3(4.0)] → 2 items
  Top-2 predicted: [item2(4.9), item3(4.5)]
  Hits:           2
  Precision@2:    2/2 = 1.0
  Recall@2:       2/2 = 1.0

User 2:
  Relevant items:  [item3(5.0), item4(4.0)] → 2 items
  Top-2 predicted: [item3(4.8), item2(4.2)]
  Hits:           1 (only item3)
  Precision@2:    1/2 = 0.5
  Recall@2:       1/2 = 0.5

Expected Avg Precision@2: 0.750
Computed Avg Precision@2: 0.750
✓ Match: True

Expected Avg Recall@2:    0.750
Computed Avg Recall@2:    0.750
✓ Match: True

✅ Multi-user averaging works correctly!


## Summary

All metric implementations have been validated with toy examples:

✅ **RMSE**: Correctly computes root mean squared error
✅ **MAD**: Correctly computes mean absolute deviation
✅ **Precision@N**: Correctly identifies relevant items in top-N recommendations
✅ **Recall@N**: Correctly measures coverage of relevant items
✅ **Edge cases**: Handles N > candidates correctly
✅ **Multi-user**: Correctly averages metrics across users

The implementation in `main_experiment_grid_search_1229.ipynb` is correct!